# Season-level recurrence: was 2026 exceptional because ridges kept regenerating, not just because one ran long?

**Motivation.** Every mechanism notebook so far (`CA_ridging_persistence_drivers.ipynb`,
`CA_ridging_persistence_drivers_upstream.ipynb`, `CA_ridging_persistence_drivers_waf.ipynb`,
`CA_ridging_persistence_background_state.ipynb`) treats the *episode* as the unit of
analysis -- what made March 2026's 19 days long. Seven tests of that question came back
null or non-explanatory. A different framing: WY2026 had **four** Z500 episodes
(Jan, Late Jan/Feb, Late Feb, March) packed into a 79-day span with gaps of only 11, 18,
and 10 days between them -- maybe the anomaly isn't one episode's internal dynamics, but
the *season* regenerating ridges faster than normal, leaving no recovery window. A
persistently favorable background state (PDO, found to be the strongest predictor in
this thread so far, `CA_ridging_persistence_background_state.ipynb`) would produce
exactly this signature -- not one slower-decaying episode, but a whole season tilted
toward frequent, closely-spaced ridging.

This notebook tests that directly at the **season** level (one row per water year, not
per episode): episode count, total days spent in an active episode, and the gaps
between consecutive episodes within a season, for the full 1990-2025 record plus 2026.
No new fetch -- reuses the fit-period event CSV and the PDO/ENSO indices already
fetched for `CA_ridging_persistence_background_state.ipynb`.

**Caveats carried over from that notebook:** PDO has no 2026 data (NOAA PSL's series
ends August 2025), so 2026 can only be located within the ONI, not PDO, season-level
distribution. Season-level aggregates also mean a much smaller sample for the
gap-based statistics specifically (only seasons with >=2 episodes have a gap to
measure at all) -- reported explicitly below, not glossed over.

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date
from scipy.stats import spearmanr, percentileofscore
import sys

import os
os.environ.setdefault('PROJ_DATA', str(Path(sys.prefix) / 'share' / 'proj'))

sys.path.insert(0, str(Path.cwd()))
import ca_blocking_lib as lib

DATA_DIR = Path('/glade/campaign/mmm/c3we/mingge/ERA5/CA_blocking')
FIT_YEARS = list(range(1990, 2026))
YEAR_2026 = 2026
THRESHOLD_SD = 1
VARS = [('z500', '500 hPa height (ridging)'), ('t700', '700 hPa temperature (warm)')]
COLOR = {'500 hPa height (ridging)': '#2a78d6', '700 hPa temperature (warm)': '#eb6834'}
N_BOOT = 2000
rng = np.random.default_rng(42)

CLIM_WINDOW = 5
MIN_DURATION = 5
MAX_GAP = 1
SIGN = 1


## 1. Load the full-record Method A event table

Same `ca_z500_t700_events_1990-2025_detrended_mindur5.csv` used throughout this
mechanism thread.

In [2]:
events = pd.read_csv(DATA_DIR / 'ca_z500_t700_events_1990-2025_detrended_mindur5.csv',
                     parse_dates=['start_date', 'end_date'])
events['start_date'] = events.start_date.dt.date
events['end_date'] = events.end_date.dt.date
events = events[events.threshold_sd == THRESHOLD_SD].reset_index(drop=True)
events['year'] = events.start_date.apply(lambda d: d.year)

print(f'{len(events)} events at >= {THRESHOLD_SD} sigma, by variable:')
print(events.groupby('variable').size())


95 events at >= 1 sigma, by variable:
variable
500 hPa height (ridging)      48
700 hPa temperature (warm)    47
dtype: int64


## 2. Reproduce the four WY2026 episodes -- both Z500 and T700

Live from the frozen 1990-2025 baseline (not hardcoded), same pattern as every prior
mechanism notebook. Earlier notebooks only needed the Z500 side; this one needs both,
since clustering is tested for both variables.

In [3]:
def reproduce_2026_episodes(varname, label):
    files = [DATA_DIR / f'z500_t700_ca_{y}.nc' for y in FIT_YEARS]
    ds = xr.open_mfdataset(files, combine='by_coords', engine='netcdf4')
    weights = np.cos(np.deg2rad(ds.latitude))
    fld = ds[varname].weighted(weights).mean(dim=('latitude', 'longitude')).compute()
    df = pd.DataFrame({varname: fld.values}, index=pd.to_datetime(ds.time.values)).sort_index()
    df = df[df.index.month.isin([1, 2, 3, 4])]
    sanom, diag = lib.standardize_with_trend(df.index, df[varname].values, window=CLIM_WINDOW,
                                              label=varname, verbose=False)

    ds26 = xr.open_dataset(DATA_DIR / 'z500_t700_ca_2026.nc', engine='netcdf4')
    weights26 = np.cos(np.deg2rad(ds26.latitude))
    fld26 = ds26[varname].weighted(weights26).mean(dim=('latitude', 'longitude')).compute()
    df26 = pd.DataFrame({varname: fld26.values}, index=pd.to_datetime(ds26.time.values)).sort_index()
    df26 = df26[df26.index.month.isin([1, 2, 3, 4])]
    clim_mean_arr = lib.lookup_climatology(df26.index, diag['clim_mean_dict'])
    clim_std_arr = lib.lookup_climatology(df26.index, diag['clim_std_dict'])
    trend_2026 = diag['trend']['intercept'] + diag['trend']['slope'] * 2026
    df26['sanom'] = (df26[varname] - (clim_mean_arr + trend_2026)) / clim_std_arr

    # [Fixed 2026-11-04] was hardcoded threshold=1, silently ignoring THRESHOLD_SD --
    # harmless while THRESHOLD_SD == 1, but would have silently decoupled this
    # episode-detection call from the historical `events` table a few cells up (which
    # correctly filters on `events.threshold_sd == THRESHOLD_SD`) the moment someone
    # changed THRESHOLD_SD to test sensitivity.
    ep = lib.find_events(df26.index, df26['sanom'].values, threshold=THRESHOLD_SD,
                          min_duration=MIN_DURATION, sign=SIGN, max_gap=MAX_GAP)
    ep = ep[['start_date', 'end_date', 'duration_days', 'peak_sanom']].reset_index(drop=True)
    ep['variable'] = label
    ep['year'] = YEAR_2026
    return ep


episodes_2026 = pd.concat([reproduce_2026_episodes(v, label) for v, label in VARS], ignore_index=True)
print('WY2026 episodes, both variables:')
print(episodes_2026[['variable', 'start_date', 'end_date', 'duration_days']].to_string(index=False))


WY2026 episodes, both variables:
                  variable start_date   end_date  duration_days
  500 hPa height (ridging) 2026-01-10 2026-01-18              9
  500 hPa height (ridging) 2026-01-29 2026-02-05              8
  500 hPa height (ridging) 2026-02-23 2026-03-01              7
  500 hPa height (ridging) 2026-03-11 2026-03-29             19
700 hPa temperature (warm) 2026-01-12 2026-01-21             10
700 hPa temperature (warm) 2026-01-30 2026-02-05              7
700 hPa temperature (warm) 2026-02-23 2026-03-04             10
700 hPa temperature (warm) 2026-03-12 2026-03-30             19


## 3. Season-level aggregates: episode count, total days, and inter-episode gaps

One row per (year, variable): number of episodes, total days spent in an episode,
fraction of the ~120-day Jan-Apr season covered, the single longest episode, and
(for seasons with >=2 episodes) the minimum and median gap between consecutive
episode windows -- the direct operationalization of "no chance to recover."

In [4]:
def jan_apr_days(year):
    return (date(year, 4, 30) - date(year, 1, 1)).days + 1


def season_aggregate(sub, year, label):
    sub = sub.sort_values('start_date').reset_index(drop=True)
    n = len(sub)
    total_days = jan_apr_days(year)
    gaps = [(sub.start_date[i + 1] - sub.end_date[i]).days - 1 for i in range(n - 1)]
    return {
        'year': year, 'variable': label, 'n_episodes': n,
        'total_episode_days': int(sub.duration_days.sum()) if n else 0,
        'frac_season_covered': sub.duration_days.sum() / total_days if n else 0.0,
        'max_duration': int(sub.duration_days.max()) if n else 0,
        'min_gap': min(gaps) if gaps else np.nan,
        'median_gap': float(np.median(gaps)) if gaps else np.nan,
        'n_gaps': len(gaps),
    }


all_events = pd.concat([events[['variable', 'year', 'start_date', 'end_date', 'duration_days']],
                         episodes_2026[['variable', 'year', 'start_date', 'end_date', 'duration_days']]],
                        ignore_index=True)

season_rows = []
for col, label in VARS:
    for y in FIT_YEARS + [YEAR_2026]:
        sub = all_events[(all_events.variable == label) & (all_events.year == y)]
        season_rows.append(season_aggregate(sub, y, label))
season_df = pd.DataFrame(season_rows)

n_multi = (season_df[season_df.year.isin(FIT_YEARS)]
           .groupby('variable')['n_episodes'].apply(lambda s: (s >= 2).sum()))
print('Seasons (of 36 fit-period years) with >=2 episodes, i.e. with a measurable gap:')
print(n_multi)
season_df[season_df.year == 2026]


Seasons (of 36 fit-period years) with >=2 episodes, i.e. with a measurable gap:
variable
500 hPa height (ridging)      16
700 hPa temperature (warm)    18
Name: n_episodes, dtype: int64


,year,variable,n_episodes,total_episode_days,frac_season_covered,max_duration,min_gap,median_gap,n_gaps
36,2026,500 hPa height (ridging),4,43,0.358333,19,9.0,10.0,3
73,2026,700 hPa temperature (warm),4,46,0.383333,19,7.0,8.0,3


## 4. Where does WY2026 sit in the season-level distribution?

Percentile rank within the 1990-2025 fit-period seasons, for each variable, on every
clustering metric -- episode count, total days covered, and gap structure. For the
gap metrics, a *low* percentile means 2026's gaps are *shorter* than most historical
seasons (i.e. more clustered), stated explicitly below rather than left to
percentile-direction ambiguity.

In [5]:
CLUSTER_METRICS = ['n_episodes', 'total_episode_days', 'frac_season_covered', 'max_duration',
                   'min_gap', 'median_gap']

for col, label in VARS:
    fit = season_df[(season_df.variable == label) & (season_df.year.isin(FIT_YEARS))]
    row2026 = season_df[(season_df.variable == label) & (season_df.year == YEAR_2026)].iloc[0]
    print(f'--- {label} ---')
    for m in CLUSTER_METRICS:
        valid = fit[m].dropna()
        val = row2026[m]
        if pd.isna(val) or len(valid) < 8:
            print(f'  {m}: 2026={val} (n/a or insufficient fit-period seasons with this metric, n={len(valid)})')
            continue
        pct = percentileofscore(valid, val)
        direction = ''
        if m in ('min_gap', 'median_gap'):
            direction = f' -- shorter than {100 - pct:.0f}% of multi-episode fit-period seasons' if pct < 50 else                         f' -- longer than {pct:.0f}% of multi-episode fit-period seasons'
        print(f'  {m}: 2026={val:.2f}, {pct:.0f}th percentile of {len(valid)} fit-period seasons{direction}')
    print()


--- 500 hPa height (ridging) ---
  n_episodes: 2026=4.00, 100th percentile of 36 fit-period seasons
  total_episode_days: 2026=43.00, 100th percentile of 36 fit-period seasons
  frac_season_covered: 2026=0.36, 100th percentile of 36 fit-period seasons
  max_duration: 2026=19.00, 100th percentile of 36 fit-period seasons
  min_gap: 2026=9.00, 12th percentile of 16 fit-period seasons -- shorter than 88% of multi-episode fit-period seasons
  median_gap: 2026=10.00, 12th percentile of 16 fit-period seasons -- shorter than 88% of multi-episode fit-period seasons

--- 700 hPa temperature (warm) ---
  n_episodes: 2026=4.00, 100th percentile of 36 fit-period seasons
  total_episode_days: 2026=46.00, 100th percentile of 36 fit-period seasons
  frac_season_covered: 2026=0.38, 100th percentile of 36 fit-period seasons
  max_duration: 2026=19.00, 100th percentile of 36 fit-period seasons
  min_gap: 2026=7.00, 11th percentile of 18 fit-period seasons -- shorter than 89% of multi-episode fit-period 

## 5. Does season-level PDO/ENSO predict clustering (not just single-episode duration)?

`CA_ridging_persistence_background_state.ipynb` found PDO predicts *single-episode*
Z500 duration (rho=0.46, p=0.0009, Bonferroni-robust). This tests a different claim:
does the *season-mean* PDO/ENSO state predict how many episodes occur and how tightly
they cluster -- using the same ONI/PDO data already fetched
(`get_mjo_enso_pdo_indices.py`), aggregated to a Jan-Apr seasonal mean per year instead
of an event-window mean.

In [6]:
oni = pd.read_csv(DATA_DIR / 'oni_index.csv')
pdo = pd.read_csv(DATA_DIR / 'pdo_index.csv')

def jan_apr_mean(df, valcol, year):
    sub = df[(df.year == year) & (df.month.isin([1, 2, 3, 4]))]
    return float(sub[valcol].mean()) if sub[valcol].notna().any() else np.nan

season_df['enso_oni_season'] = season_df.year.apply(lambda y: jan_apr_mean(oni, 'oni', y))
season_df['pdo_season'] = season_df.year.apply(lambda y: jan_apr_mean(pdo, 'pdo', y))

print(f"2026 season-mean ONI: {season_df[season_df.year==2026].enso_oni_season.iloc[0]:.2f}")
print(f"2026 season-mean PDO: {season_df[season_df.year==2026].pdo_season.iloc[0]} (NaN expected -- no 2026 PDO data)")


2026 season-mean ONI: -0.01
2026 season-mean PDO: nan (NaN expected -- no 2026 PDO data)


In [7]:
def bootstrap_spearman_ci(x, y, n_boot=N_BOOT, ci=90, rng=rng):
    n = len(x)
    boot = np.empty(n_boot)
    idx_pool = np.arange(n)
    for b in range(n_boot):
        idx = rng.choice(idx_pool, size=n, replace=True)
        boot[b] = spearmanr(x[idx], y[idx]).statistic
    lo, hi = np.percentile(boot, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return lo, hi


BG_PREDICTORS = ['enso_oni_season', 'pdo_season']
CLUSTER_TARGETS = ['n_episodes', 'total_episode_days', 'min_gap']

results = []
for col, label in VARS:
    fit = season_df[(season_df.variable == label) & (season_df.year.isin(FIT_YEARS))]
    for target in CLUSTER_TARGETS:
        for pred in BG_PREDICTORS:
            valid = fit[[target, pred]].dropna()
            x, y = valid[pred].values, valid[target].values
            if len(valid) < 8:
                results.append({'variable': label, 'target': target, 'predictor': pred,
                                 'n': len(valid), 'rho': np.nan, 'p_spearman': np.nan,
                                 'ci_lo': np.nan, 'ci_hi': np.nan})
                continue
            rho, p = spearmanr(x, y)
            lo, hi = bootstrap_spearman_ci(x, y)
            results.append({'variable': label, 'target': target, 'predictor': pred, 'n': len(valid),
                             'rho': rho, 'p_spearman': p, 'ci_lo': lo, 'ci_hi': hi})

results_df = pd.DataFrame(results)
pd.set_option('display.width', 120)
results_df


,variable,target,predictor,n,rho,p_spearman,ci_lo,ci_hi
0,500 hPa height (ridging),n_episodes,enso_oni_season,36,-0.050777,0.768680,-0.343642,0.258258
1,500 hPa height (ridging),n_episodes,pdo_season,36,0.071147,0.680089,-0.245544,0.406886
2,500 hPa height (ridging),total_episode_days,enso_oni_season,36,0.060846,0.724447,-0.269222,0.354195
3,500 hPa height (ridging),total_episode_days,pdo_season,36,0.180674,0.291657,-0.161560,0.491438
4,500 hPa height (ridging),min_gap,enso_oni_season,16,-0.485294,0.056718,-0.805175,-0.007493
5,500 hPa height (ridging),min_gap,pdo_season,16,-0.438235,0.089530,-0.746666,-0.011682
6,700 hPa temperature (warm),n_episodes,enso_oni_season,36,-0.034913,0.839801,-0.341107,0.269313
7,700 hPa temperature (warm),n_episodes,pdo_season,36,0.189541,0.268213,-0.106899,0.470420
8,700 hPa temperature (warm),total_episode_days,enso_oni_season,36,0.015843,0.926931,-0.292253,0.330337
9,700 hPa temperature (warm),total_episode_days,pdo_season,36,0.245448,0.149050,-0.058433,0.527549


## 6. Conclusion

**WY2026 is an unambiguous, literal record-holder at the season level -- the clearest,
least caveated finding in this entire mechanism thread -- but the one real background-
state relationship found for clustering still doesn't explain why 2026 itself clustered
so extremely.**

- **2026 holds the all-time record (100th percentile of 36 fit-period seasons) on
  every season-coverage metric, for both Z500 and T700.** 4 episodes, 43-46 total
  episode-days, and 36-38% of the entire Jan-Apr season spent in an active >=1-sigma
  episode -- more than any single year in the 1990-2025 record on every one of these
  measures, not just an outlier on one axis. This is a genuinely different and much
  less ambiguous statement than anything found in prior notebooks: the "multiple major
  warm ridges" framing isn't just true, it's the most extreme season in 36 years by
  this measure, full stop.
- **Its inter-episode gaps are compressed into the tight tail, confirming "no chance to
  recover" quantitatively.** Z500 gaps of 9-10 days sit at the 12th percentile of the
  16 fit-period seasons that had >=2 episodes to measure a gap from at all (shorter
  than 88% of them); T700's 7-8 day gaps sit at the 6th-11th percentile (shorter than
  89-94%). Not literally the single tightest season ever recorded, but unambiguously in
  the most-clustered tail of seasons that ever had multiple episodes to cluster.
- **Season-mean ENSO and PDO show a real, coherent relationship to clustering tightness
  -- a second, independent manifestation of the same background-state story found for
  single-episode duration.** `CA_ridging_persistence_background_state.ipynb` found PDO
  predicts how *long* a single episode lasts; this notebook finds season-mean ENSO and
  PDO also predict how *tightly episodes cluster* (`min_gap`), a different quantity
  from duration. T700 min_gap vs. ENSO: rho=-0.61, p=0.0078 (n=18); vs. PDO: rho=-0.48,
  p=0.042 (n=18). Z500 shows the same direction, weaker: ENSO rho=-0.49, p=0.057; PDO
  rho=-0.44, p=0.090 (both n=16). The negative sign means a warmer/more El-Nino-like
  state predicts *shorter* gaps -- the same direction as the single-episode PDO finding,
  not a new, independent one -- consistent with one underlying "warm background state
  favors frequent, closely-spaced ridging" story rather than two unrelated results.
  None of these clear a strict 12-test Bonferroni bar (p<0.0042), and the small sample
  (n=16-18, since only seasons with >=2 episodes have a gap to measure) makes this more
  fragile than the single-episode PDO result -- suggestive and coherent, not confirmed.
  Episode *count* and *total days*, by contrast, show no relationship to ENSO/PDO at
  all (p=0.15-0.93) -- background state, if it matters here, acts specifically on
  spacing, not on how often ridging happens overall.
- **2026 itself does not have an anomalous background state, so this still doesn't
  explain 2026 specifically.** 2026's season-mean ONI was -0.01 -- almost exactly
  neutral, not the El-Nino-like state that would predict unusually tight clustering
  under the relationship just found. PDO can't be checked at all (no 2026 data, same
  gap noted in the background-state notebook). So even the one coherent, physically
  sensible relationship in this analysis fails to explain why 2026's clustering was the
  most extreme in 36 years -- the same pattern every other mechanism notebook in this
  thread has found: real population-level structure, but the specific 2026 case sits
  outside what that structure predicts.

**Net effect on the open mechanism question.** This reframing was worth doing: it
establishes, for the first time with no caveats, that 2026 was record-breaking at the
season level (not just "March happened to be long") and that background state relates
to clustering as well as duration. But the central mystery is unchanged -- nothing
tested so far, across eight notebooks and roughly a dozen distinct hypotheses, explains
why *2026 specifically* produced this record season rather than merely showing that the
record exists and has some statistical correlates. The clustering angle sharpens what
needs explaining (a season-long regenerative tendency, not a single episode's decay)
without yet explaining it.
